# Lab 8 - 通常 Agent の sequential workflow を Hosted Agent にする

入力整理、規程確認、最終レビューを担当する 3 つの通常 Agent を順番につなぎます。participant 間の引き継ぎをコード・グラフ・途中回答で確認してから、同じ checked-in source を Microsoft Foundry の Hosted Agent として deploy します。

```text
intake_agent -> policy_agent -> reviewer_agent
```

**Lab 7 の Notebook 実行結果には依存しません。** Harness Agent は tool と Skill を組み合わせる Lab 7 に限定します。Lab 8 は Luna の token 消費を抑えるため、通常 Agent と Lab 3 の Foundry IQ だけを使います。

**使用する kernel:** `Python (Foundry Hosted Agent)`。Codespaces またはローカルの同じ Dev Container で `00-setup.ipynb` を完了してから開きます。Graphviz と kernel はコンテナー作成時に準備済みです。

> **境界と料金:** モデル利用料金に加え、Foundry IQ、source remote build、Hosted Agent の稼働には別途の実行料金が発生します。架空のデータだけを使ってください。デプロイ・削除セルには、それぞれ確認入力が必要です。

## 1. Hosted workflow の接続先を読み込む

`00-setup.ipynb` が生成した `.workshop/context.json` から model と Search endpoint を取得し、Lab 3 の Foundry IQ 名を設定します。`FOUNDRY_PROJECT_ENDPOINT` はローカル実行用です。deploy 後は Hosted Agent platform が自動注入します。

In [ ]:
import os
import sys
from pathlib import Path


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src" / "hosted-agent" / "workflow.py").is_file():
            return candidate
    raise RuntimeError(
        "Repository root が見つかりません。clone した教材内の Notebook を開いてください。"
    )


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.lib.workshop_context import load_context, validate_workshop_context  # noqa: E402
from scripts.lib.workshop_runtime import HOSTED_ENVIRONMENT, require_current_runtime  # noqa: E402

require_current_runtime(HOSTED_ENVIRONMENT)
context_path = REPO_ROOT / ".workshop" / "context.json"
if not context_path.is_file():
    raise FileNotFoundError("00-setup.ipynb を完了し、.workshop/context.json を作成してください。")

context = validate_workshop_context(load_context(context_path))
outputs = context["resource_outputs"]
os.environ["WORKSHOP_CREDENTIAL_MODE"] = "azure-cli"
os.environ["FOUNDRY_PROJECT_ENDPOINT"] = outputs["foundry_project_endpoint"]["value"]
os.environ["FOUNDRY_MODEL"] = outputs["primary_model_deployment_name"]["value"]
os.environ["AZURE_AI_SEARCH_SERVICE_ENDPOINT"] = outputs["search_service_endpoint"]["value"]
os.environ["AZURE_AI_SEARCH_KNOWLEDGE_BASE_NAME"] = "contoso-travel-knowledge-lab"

hosted_source = REPO_ROOT / "src" / "hosted-agent"
if str(hosted_source) not in sys.path:
    sys.path.insert(0, str(hosted_source))

print(f"Project: {outputs['foundry_project_name']['value']}")
print(f"Model: {os.environ['FOUNDRY_MODEL']}")
print(f"Foundry IQ: {os.environ['AZURE_AI_SEARCH_KNOWLEDGE_BASE_NAME']}")

## 2. 役割の異なる 3 participant を作る

| Participant | 担当 | 持つ機能 |
|---|---|---|
| `intake_agent` | 依頼の値と不足項目を整理 | model + instructions |
| `policy_agent` | Foundry IQ で社内規程を検索し、文書 ID と出典を整理 | model + Foundry IQ |
| `reviewer_agent` | 元の依頼と根拠を照合し、最終回答に整える | model + instructions |

3 participant はすべて通常の `Agent` です。Lab 7 の Harness、todo、memory、Toolbox Skills は使いません。複雑な 1 Agent に仕事を集めず、規程確認だけを `policy_agent` に分担することで Luna の token 消費を抑えます。

In [ ]:
from contextlib import AsyncExitStack

import travel_agents
import workflow

credential = travel_agents.create_credential()
chat_client = travel_agents.create_chat_client(credential)
foundry_iq_tool = travel_agents.create_foundry_iq_tool(credential)
intake_agent = chat_client.as_agent(
    name="intake_agent",
    instructions=workflow.INTAKE_AGENT_INSTRUCTIONS,
)
policy_agent = chat_client.as_agent(
    name="policy_agent",
    instructions=workflow.POLICY_AGENT_INSTRUCTIONS,
    tools=[foundry_iq_tool],
)
reviewer_agent = chat_client.as_agent(
    name="reviewer_agent",
    instructions=workflow.REVIEWER_AGENT_INSTRUCTIONS,
)
participants = [intake_agent, policy_agent, reviewer_agent]
expected_order = [agent.name for agent in participants]
print(" -> ".join(expected_order))

resource_stack = AsyncExitStack()
await resource_stack.__aenter__()
for participant in participants:
    await resource_stack.enter_async_context(participant)

## 3. `SequentialBuilder` で順番を固定する

`SequentialBuilder` は複数の通常 Agent の実行順と会話の引き継ぎを定義します。元の依頼と、それまでの participant の回答が次へ渡ります。

`output_from=[reviewer_agent]` で最終回答を reviewer に限定し、Notebook だけ `intermediate_output_from="all_other"` を使って intake / policy の途中回答を観察します。deploy 用 source は reviewer の最終回答だけを公開します。

In [ ]:
from agent_framework import Workflow
from agent_framework.orchestrations import SequentialBuilder


def build_travel_workflow() -> Workflow:
    return SequentialBuilder(
        participants=participants,
        output_from=[reviewer_agent],
        intermediate_output_from="all_other",
    ).build()


travel_workflow = build_travel_workflow()

## 4. 実際の workflow graph を表示する

`WorkflowViz` は今作った workflow object から graph を生成します。`intake_agent -> policy_agent -> reviewer_agent` の順を確認してください。

In [ ]:
import shutil
import subprocess

from agent_framework import WorkflowViz
from IPython.display import SVG, Code, display

viz = WorkflowViz(travel_workflow)
mermaid_graph = viz.to_mermaid()
dot_executable = shutil.which("dot")
if dot_executable is None:
    print("Graphviz がないため Mermaid 定義を表示します。")
    display(Code(mermaid_graph, language="text"))
else:
    rendered = subprocess.run(
        [dot_executable, "-Tsvg"],
        input=viz.to_digraph(),
        capture_output=True,
        text=True,
        encoding="utf-8",
        check=False,
    )
    if rendered.returncode != 0:
        print(rendered.stderr)
        rendered.check_returncode()
    display(SVG(data=rendered.stdout))

## 5. 途中回答と最終回答を観察する

標準の合成依頼は、国内出張の日当、宿泊上限、精算期限を文書 ID またはリンク付きで確認します。費用見積もり、予約、申請、承認、精算は明示的に対象外です。

stream event を participant ごとに分けて表示します。`policy_agent` の `knowledge_base_retrieve` と participant 間の引き継ぎに注目します。

In [ ]:
from agent_framework import AgentResponseUpdate
from IPython.display import Markdown, display

user_request = workflow.SAMPLE_REQUEST
execution_order = []
intermediate_answers = {}
policy_actions = set()
final_chunks = []
answer = None
travel_workflow = build_travel_workflow()

async for event in travel_workflow.run(user_request, stream=True):
    if event.type in {"failed", "executor_failed", "error"}:
        raise RuntimeError(f"Workflow failed: {event.details or event.data}")
    if event.executor_id == policy_agent.name and isinstance(event.data, AgentResponseUpdate):
        policy_actions.update(
            content.name for content in event.data.contents if getattr(content, "name", None)
        )
    if event.type == "executor_invoked" and event.executor_id in expected_order:
        execution_order.append(event.executor_id)
        print(f"\n開始: {event.executor_id}", flush=True)
    elif event.type == "intermediate" and isinstance(event.data, AgentResponseUpdate):
        intermediate_answers.setdefault(event.executor_id, "")
        intermediate_answers[event.executor_id] += event.data.text
        print(event.data.text, end="", flush=True)
    elif event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        final_chunks.append(event.data.text)

answer = "".join(final_chunks)
if not answer.strip():
    raise RuntimeError("reviewer の最終回答が空です。実行ログを確認してください。")
display(Markdown("### reviewer_agent の最終回答"))
print(answer)
print("Policy actions:", sorted(policy_actions))

In [ ]:
assert execution_order == expected_order, execution_order
assert set(intermediate_answers) == {"intake_agent", "policy_agent"}
assert all(intermediate_answers.values())
assert "knowledge_base_retrieve" in policy_actions, (
    "policy_agent の Foundry IQ 実行を確認してください。",
    policy_actions,
)
unexpected_actions = {"load_skill", "tool_search", "call_tool", "createTripEstimate"}
assert not unexpected_actions.intersection(policy_actions), policy_actions
assert all(heading in answer for heading in ["依頼の整理", "規程確認", "次のアクション"])
assert workflow.SIMULATION_NOTICE in answer
print("OK: participant 順序、途中回答、reviewer の最終形式を確認しました。")

## 6. Notebook と deploy source の対応を確認する

Notebook は学習のため participant 作成を展開しました。deploy 対象は `src/hosted-agent/` です。次の source が同じ 3 つの通常 Agent と participant 順を使うことを確認します。

In [ ]:
import inspect

display(Code(inspect.getsource(workflow.build_workflow), language="python"))
display(Code(workflow.POLICY_AGENT_INSTRUCTIONS, language="text"))

## 7. Azure を使わない contract test を実行する

実モデルの回答は変動します。fake client を使うテストでは、3 つの通常 Agent の順序、会話の引き継ぎ、Foundry IQ が policy agent だけに接続されること、reviewer だけが最終回答になることを固定して確認します。

In [ ]:
completed = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/contract/hosted_agent/test_sequential_workflow.py",
        "tests/contract/hosted_agent/test_telemetry.py",
        "-q",
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    check=False,
)
print(completed.stdout)
if completed.returncode != 0:
    print(completed.stderr)
    completed.check_returncode()

## 8. ローカル接続を閉じる

Foundry IQ の MCP session と HTTP client を閉じます。Azure resource は削除しません。

In [ ]:
await resource_stack.aclose()
credential.close()
print("Local workflow resources closed.")

## 9. Hosted Agent を deploy する

以下のセルは、`WORKSHOP_MANAGEMENT_PYTHON` が指す管理用 `venv` (既定値 `~/.venvs/foundry-workshop/bin/python`) からデプロイします。Python 3.12 の所有・version を検証し、Notebook の Hosted 用 SDK と混在させません。

確認欄へ `DEPLOY` と入力すると、`src/hosted-agent/` の source remote build と起動を開始します。実行中は再送しません。

`status: "active"` を確認したら [Lab 8 の Portal 手順](../labs/08-hosted-multi-agent.md#3-portal-で実行する)で実際の応答を確認してください。

In [ ]:
import subprocess

from scripts.lib.workshop_context import load_context, validate_workshop_context
from scripts.lib.workshop_runtime import WORKSHOP_ENVIRONMENT, get_runtime_python

HOSTED_AGENT_NAME = "contoso-travel-hosted-planner"


def run_management_script(script_name: str, *arguments: str) -> None:
    management_python = get_runtime_python(WORKSHOP_ENVIRONMENT)
    current_context = validate_workshop_context(
        load_context(context_path),
        expected_subscription_id=context["subscription_id"],
        expected_resource_group=context["resource_group_name"],
    )
    if current_context != context:
        raise RuntimeError("接続設定が変更されています。先頭から接続先を確認し直してください。")
    result = subprocess.run(
        [
            str(management_python),
            str(REPO_ROOT / "scripts" / script_name),
            "--context",
            str(context_path),
            *arguments,
        ],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

In [ ]:
try:
    confirmation = input("Hosted Agent を作成して起動する場合は DEPLOY と入力: ").strip()
except (EOFError, KeyboardInterrupt):
    confirmation = ""
if confirmation == "DEPLOY":
    run_management_script(
        "deploy_hosted_agent.py",
        "--agent-name",
        HOSTED_AGENT_NAME,
        "--source-dir",
        str(REPO_ROOT / "src" / "hosted-agent"),
        "--output",
        "json",
    )
else:
    print("デプロイは実行していません。")

## 10. Lab 9 で Hosted Agent を削除する

**Foundry Playground での応答・Trace 確認と必要な Export が終わってから**実行します。確認欄へ表示された Agent 名を入力すると、この教材の接続先にある Agent と全 version を削除します。Codespace の停止・削除と RG の削除は [Lab 9](../labs/09-observability-cleanup.md) に従ってください。

In [ ]:
try:
    confirmation = input(f"削除する場合は Agent 名 {HOSTED_AGENT_NAME} を入力: ").strip()
except (EOFError, KeyboardInterrupt):
    confirmation = ""
if confirmation == HOSTED_AGENT_NAME:
    run_management_script(
        "delete_hosted_agent.py",
        "--agent-name",
        HOSTED_AGENT_NAME,
        "--subscription",
        context["subscription_id"],
        "--resource-group",
        context["resource_group_name"],
        "--output",
        "json",
    )
else:
    print("削除は実行していません。")